In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# kagglehub.dataset_download('<owner>/<dataset-slug>')




/kaggle/input/datasets/kmader/lungnodemalignancy/ct_tiles.tif
/kaggle/input/datasets/kmader/lungnodemalignancy/all_patches.hdf5
/kaggle/input/datasets/kmader/lungnodemalignancy/malignancy.csv


In [13]:
import h5py
import numpy as np
import pandas as pd
import torch.nn.functional as F
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split




f = h5py.File('/kaggle/input/datasets/kmader/lungnodemalignancy/all_patches.hdf5', 'r')
print(list(f.keys()))

['ct_slices', 'slice_class']


In [14]:

x= np.array(f['ct_slices'])
y=np.array(f['slice_class'])



In [15]:
np.unique(y)
print (np.sum(y == 0),np.sum(y == 1))
x=(x-np.min(x))/(np.max(x)-np.min(x))
print(x)

4165 2526
[[[0.46398687 0.4559475  0.4454471  ... 0.38113207 0.37063166 0.34273994]
  [0.45250204 0.4400328  0.42608696 ... 0.35422477 0.35602954 0.3430681 ]
  [0.43675143 0.42526662 0.4100082  ... 0.34602132 0.34946677 0.34323215]
  ...
  [0.5002461  0.497621   0.49384743 ... 0.4802297  0.48088598 0.4807219 ]
  [0.4949959  0.49351928 0.49466777 ... 0.48088598 0.48055783 0.4812141 ]
  [0.49351928 0.4949959  0.4987695  ... 0.47662017 0.47514355 0.47711238]]

 [[0.38621822 0.37916324 0.3750615  ... 0.34700575 0.34208366 0.34060705]
  [0.373749   0.36390483 0.3604594  ... 0.3435603  0.33732566 0.3388023 ]
  [0.36505333 0.35652173 0.35488105 ... 0.3414274  0.33552092 0.33519277]
  ...
  [0.49909762 0.49745694 0.497621   ... 0.48170632 0.4812141  0.47924528]
  [0.49450368 0.4964725  0.49811321 ... 0.4802297  0.47940934 0.478589  ]
  [0.4895816  0.49515995 0.49844134 ... 0.48055783 0.47990155 0.4802297 ]]

 [[0.36062345 0.3583265  0.35816243 ... 0.34323215 0.33847415 0.33863822]
  [0.35242  

In [16]:
x=x.reshape (-1,1,64,64)
print (x.shape)

(6691, 1, 64, 64)


In [17]:
x_train ,x_test ,y_train ,y_test=train_test_split (x,y,test_size=0.2,random_state=42)
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch .tensor(y_train, dtype=torch.float32)
x_test_tensor=torch.tensor(x_test,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.float32)
dataset = TensorDataset(x_train_tensor , y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [18]:
class LungCancerCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(0.5)
        self.conv1=nn.Conv2d(1,32,kernel_size=3)
        self.pool=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(32,64,kernel_size=3)
        self.conv3=nn.Conv2d(64,128,kernel_size=3)
        self.fc1=nn.Linear(4608,128)
        self.fc2=nn.Linear(128,1)
    
    def forward(self,x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.pool(x)
        x=x.view(x.size(0), -1)
        x=F.relu(self.fc1(x))
        self.dropout = nn.Dropout(0.5)
        x=torch.sigmoid(self.fc2(x))
        return x 
model = LungCancerCNN()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion=nn.BCELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.0001)
    
for epoch in range (1):
        model.train()
        
        for x_train,y_train in loader:
            x_train_tensor = x_train_tensor.to(device)
            y_train_tensor = y_train_tensor.to(device)
        
            optimizer.zero_grad()
            outputs=model(x_train_tensor)
            loss=criterion(outputs,y_train_tensor)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

        



    

Epoch 1, Loss: 0.6265


In [19]:
model.eval()
with torch.no_grad():
    outputs = model(x_test_tensor.to(device))
    predictions = (outputs > 0.5).float()
    accuracy=(predictions==y_test_tensor.to(device)).float().mean()
    print(accuracy)
    

tensor(0.6587, device='cuda:0')


In [20]:


with torch.no_grad():
     def predict(patch):
         patch = (patch - np.min(patch)) / (np.max(patch) - np.min(patch))
         patch=patch.reshape(1,1,64,64)
         patch = torch.tensor(patch, dtype=torch.float32)
         patch = patch.to(device)
         outputs=model(patch)
         if outputs>0.5:
             print("malignant")
         else:
             
             print("begnine")
predict(x_test[0])


            
        
    
    


   
    

    

    

begnine


In [2]:
print(y_test[0])

NameError: name 'y_test' is not defined